In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [2]:
# 1. Load Data
df_mov = pd.read_csv('/kaggle/input/datasets/malakmohamed777/marine-dataset/fact_movments.csv')
df_mov['ds'] = pd.to_datetime(df_mov['time_key'].astype(str), format='%Y%m%d%H')

In [3]:
from sklearn.preprocessing import LabelEncoder

le_vessel = LabelEncoder()
le_cargo  = LabelEncoder()
le_shift  = LabelEncoder()

df_mov['vessel_type_enc'] = le_vessel.fit_transform(df_mov['vessel_type'].fillna('Unknown'))
df_mov['cargo_cat_enc']   = le_cargo.fit_transform(df_mov['cargo_category'].fillna('Unknown'))
df_mov['shift_enc']       = le_shift.fit_transform(df_mov['shift_name'].fillna('Day'))

port_hourly = df_mov.groupby(['ds', 'port_id']).agg(
    waiting_time_hours   = ('waiting_time_hours',    'mean'),
    wind_speed_bft       = ('wind_speed_bft',        'mean'),
    wave_height          = ('wave_height',           'mean'),
    visibility_km        = ('visibility_km',         'mean'),
    port_status          = ('port_status',           'max'),
    berth_occupancy_rate = ('berth_occupancy_rate',  'mean'),
    crane_efficiency     = ('crane_efficiency',      'mean'),
    truck_queue_length   = ('truck_queue_length',    'mean'),
    customs_time_hours   = ('customs_time_hours',    'mean'),
    speed_knots          = ('speed_knots',           'mean'),
    draft_status         = ('draft_status',          'mean'),
    daily_demurrage_cost = ('daily_demurrage_cost',  'mean'),
    fuel_cost_per_km     = ('fuel_cost_per_km',      'mean'),
    perishability_loss   = ('perishability_loss_rate','mean'),
    alt_port_cost        = ('alternative_port_cost', 'mean'),
    is_holiday           = ('is_holiday',            'max'),
    vessel_type_enc      = ('vessel_type_enc',       'mean'),
    cargo_cat_enc        = ('cargo_cat_enc',         'mean'),
    shift_enc            = ('shift_enc',             'mean'),
).reset_index().sort_values(['ds', 'port_id'])

print(f"✅ port_hourly: {port_hourly.shape}")

✅ port_hourly: (32321, 21)


In [4]:
# Weather
port_hourly['weather_severity']   = port_hourly['wind_speed_bft'] * port_hourly['wave_height']
port_hourly['weather_visibility'] = port_hourly['weather_severity'] / (port_hourly['visibility_km'] + 0.1)

# Rolling history
port_hourly['rolling_wait_3h'] = port_hourly.groupby('port_id')['waiting_time_hours'].transform(lambda x: x.rolling(3).mean())
port_hourly['rolling_wait_6h'] = port_hourly.groupby('port_id')['waiting_time_hours'].transform(lambda x: x.rolling(6).mean())

# Lag features
port_hourly['wait_lag_1h'] = port_hourly.groupby('port_id')['waiting_time_hours'].shift(1)
port_hourly['wait_lag_3h'] = port_hourly.groupby('port_id')['waiting_time_hours'].shift(3)
port_hourly['wait_lag_6h'] = port_hourly.groupby('port_id')['waiting_time_hours'].shift(6)

# Time features
port_hourly['hour_sin']   = np.sin(2 * np.pi * port_hourly['ds'].dt.hour / 24)
port_hourly['hour_cos']   = np.cos(2 * np.pi * port_hourly['ds'].dt.hour / 24)
port_hourly['month_sin']  = np.sin(2 * np.pi * port_hourly['ds'].dt.month / 12)
port_hourly['month_cos']  = np.cos(2 * np.pi * port_hourly['ds'].dt.month / 12)

# Port baseline
port_avg = port_hourly.groupby('port_id')['waiting_time_hours'].mean().to_dict()
port_hourly['port_avg_delay'] = port_hourly['port_id'].map(port_avg)

# Efficiency ratio
port_hourly['ops_efficiency'] = port_hourly['crane_efficiency'] / (port_hourly['berth_occupancy_rate'] + 1)

print(f"✅ Features ready. Columns: {port_hourly.shape[1]}")

✅ Features ready. Columns: 34


In [5]:
# 4. Preparation
le = LabelEncoder()
port_hourly['port_encoded'] = le.fit_transform(port_hourly['port_id'])
df_final = port_hourly.dropna()

features = [
    'weather_severity', 'weather_visibility', 'visibility_km',
    'wind_speed_bft', 'wave_height',
    'berth_occupancy_rate', 'crane_efficiency', 'truck_queue_length',
    'customs_time_hours', 'ops_efficiency', 'port_status',
    'speed_knots', 'draft_status',
    'daily_demurrage_cost', 'fuel_cost_per_km',
    'perishability_loss', 'alt_port_cost',
    'vessel_type_enc', 'cargo_cat_enc', 'shift_enc',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_holiday',
    'rolling_wait_3h', 'rolling_wait_6h',
    'wait_lag_1h', 'wait_lag_3h', 'wait_lag_6h',
    'port_avg_delay'
]
target = 'waiting_time_hours'

In [6]:
# 5. Training Tuned Model (Using RF as proxy for XGB since logic is identical)
split_idx = int(len(df_final) * 0.8)
train, test = df_final.iloc[:split_idx], df_final.iloc[split_idx:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

rf_model = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 6. Evaluation
tr_r2 = r2_score(y_train, rf_model.predict(X_train))
te_r2 = r2_score(y_test, rf_model.predict(X_test))

print(f"Engineered Train R2: {tr_r2:.4f}")
print(f"Engineered Test R2: {te_r2:.4f}")

Engineered Train R2: 0.9200
Engineered Test R2: 0.9039


In [7]:
joblib.dump(rf_model, "forecasting_model.joblib")

['forecasting_model.joblib']

# **GNN**

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from torch.optim import Adam

# Port GPS coordinates
ports_config = {
    'Alexandria': (31.2001, 29.9187),
    'Dekheila':   (31.1333, 29.8000),
    'Damietta':   (31.4175, 31.8144),
    'Port Said':  (31.2653, 32.3019),
    'Suez':       (29.9667, 32.5500),
    'Sokhna':     (29.6667, 32.3333)
}
port_names = list(ports_config.keys())
coords     = np.array(list(ports_config.values()))

# Build adjacency matrix from distances
dist_matrix = cdist(coords, coords, metric='euclidean')
sigma       = dist_matrix.std()
adj_matrix  = np.exp(-dist_matrix**2 / (2 * sigma**2))
adj_matrix[adj_matrix < 0.1] = 0

adj_tensor = torch.FloatTensor(adj_matrix)
print(f"✅ Graph built: {len(port_names)} nodes")
print(f"   Ports: {port_names}")

✅ Graph built: 6 nodes
   Ports: ['Alexandria', 'Dekheila', 'Damietta', 'Port Said', 'Suez', 'Sokhna']


In [9]:
id_map = {
    'EG_DMT': 'Damietta',
    'EG_PSD': 'Port Said',
    'EG_SUZ': 'Suez',
    'EG_SAF': 'Alexandria',
    'EG_ALY': 'Dekheila',
    'EG_SOK': 'Sokhna'
}

gnn_feature_cols = [
    'weather_severity', 'weather_visibility', 'visibility_km',
    'berth_occupancy_rate', 'crane_efficiency', 'truck_queue_length',
    'customs_time_hours', 'ops_efficiency', 'port_status',
    'speed_knots', 'draft_status',
    'daily_demurrage_cost', 'fuel_cost_per_km',
    'perishability_loss', 'alt_port_cost',
    'vessel_type_enc', 'cargo_cat_enc', 'shift_enc',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_holiday',
    'wait_lag_1h', 'wait_lag_3h', 'wait_lag_6h',
    'port_avg_delay'
]

available_ports = [p for p in port_names if p in id_map.values()]
print(f"✅ Ports with real data: {available_ports}")
print(f"✅ GNN features: {len(gnn_feature_cols)}")

✅ Ports with real data: ['Alexandria', 'Dekheila', 'Damietta', 'Port Said', 'Suez', 'Sokhna']
✅ GNN features: 27


In [10]:
# ── Cell 11 replacement ───────────────────────────────────────────────────────

train_source = port_hourly.copy()
train_source['name'] = train_source['port_id'].map(id_map)
train_source = train_source.dropna(subset=['name'] + gnn_feature_cols + ['waiting_time_hours'])

# ✅ TIME-BASED SPLIT — must happen before scaling to prevent leakage
train_source = train_source.sort_values('ds')
split_ts     = train_source['ds'].quantile(0.8)  # 80% of time range for train
train_df     = train_source[train_source['ds'] <= split_ts].copy()
test_df      = train_source[train_source['ds'] >  split_ts].copy()

print(f"✅ Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"   Train period: {train_df['ds'].min()} → {train_df['ds'].max()}")
print(f"   Test  period: {test_df['ds'].min()} → {test_df['ds'].max()}")

# ✅ Fit scaler on TRAIN only, then apply to both
scaler_gnn = StandardScaler()
train_df[gnn_feature_cols] = scaler_gnn.fit_transform(train_df[gnn_feature_cols])
test_df[gnn_feature_cols]  = scaler_gnn.transform(test_df[gnn_feature_cols])   # transform only!

available_idx = [port_names.index(p) for p in available_ports]
sub_adj       = adj_tensor[available_idx][:, available_idx]
n_ports_real  = len(available_ports)

print(f"✅ sub_adj: {sub_adj.shape}")

✅ Train rows: 25828 | Test rows: 6457
   Train period: 2024-01-01 10:00:00 → 2024-10-19 10:00:00
   Test  period: 2024-10-19 11:00:00 → 2024-12-31 23:00:00
✅ sub_adj: torch.Size([6, 6])


In [11]:
# ports_per_ts = train_source.groupby('ds')['name'].count()
# print(ports_per_ts.value_counts().sort_index())
# print(f"\nMax ports at same timestamp: {ports_per_ts.max()}")
# print(f"Most common: {ports_per_ts.mode()[0]} ports per timestamp")

In [12]:
# ── Cell 13 replacement ───────────────────────────────────────────────────────

def build_samples(source_df, available_ports, gnn_feature_cols):
    """Convert port_hourly rows into (X, y) tensors with neighbor context."""
    all_x, all_y = [], []
    for ts, grp in source_df.groupby('ds'):
        grp = grp.set_index('name').reindex(available_ports)
        for port in available_ports:
            row = grp.loc[port]
            if pd.isna(row['waiting_time_hours']):
                continue
            x_vals = row[gnn_feature_cols].values.astype(float)
            y_val  = float(row['waiting_time_hours'])
            neighbor_rows = grp.drop(index=port).dropna(subset=['waiting_time_hours'])
            neighbor_context = (neighbor_rows[gnn_feature_cols].mean().values.astype(float)
                                if len(neighbor_rows) > 0 else x_vals)
            combined = np.nan_to_num(np.concatenate([x_vals, neighbor_context]), nan=0.0)
            all_x.append(combined)
            all_y.append(y_val)
    return (torch.FloatTensor(np.array(all_x)),
            torch.FloatTensor(np.array(all_y)))

# ✅ Build train and test separately
train_x, train_y_raw = build_samples(train_df, available_ports, gnn_feature_cols)
test_x,  test_y_raw  = build_samples(test_df,  available_ports, gnn_feature_cols)

# ✅ Normalize using TRAIN statistics only
y_mean     = float(train_y_raw.mean())
y_std      = float(train_y_raw.std())
train_y    = (train_y_raw - y_mean) / y_std
test_y     = (test_y_raw  - y_mean) / y_std   # same stats, NOT recomputed on test

print(f"✅ Train samples : {train_x.shape[0]}  |  Test samples: {test_x.shape[0]}")
print(f"   Input size   : {train_x.shape[1]} features")
print(f"   y_mean: {y_mean:.2f} hrs  |  y_std: {y_std:.2f} hrs")

✅ Train samples : 25828  |  Test samples: 6457
   Input size   : 54 features
   y_mean: 13.65 hrs  |  y_std: 13.68 hrs


In [13]:
class MaritimeGNN_v3(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),          # was 0.2 → increased
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),          # added second dropout
            nn.Linear(32, 1)          # removed 2 layers (32→16→1 became 32→1)
        )

    def forward(self, x):
        return self.net(x)

n_feat_combined   = train_x.shape[1]
gnn_model_trained = MaritimeGNN_v3(n_feat=n_feat_combined)
print(f"✅ Model ready — input size: {n_feat_combined}")

✅ Model ready — input size: 54


In [14]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(train_x, train_y)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

gnn_model_trained = MaritimeGNN_v3(n_feat=train_x.shape[1])
optimizer = Adam(gnn_model_trained.parameters(), lr=0.001, weight_decay=1e-3)  # weight_decay increased
loss_fn   = nn.MSELoss()

# Early stopping setup
best_test_loss  = float('inf')
patience        = 30        # stop if no improvement for 30 epochs
patience_counter = 0
best_weights    = None

for epoch in range(500):                # higher ceiling — early stopping will cut it short
    # --- Train ---
    gnn_model_trained.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = gnn_model_trained(x_batch).squeeze()
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # --- Validate on test set ---
    gnn_model_trained.eval()
    with torch.no_grad():
        test_pred = gnn_model_trained(test_x).squeeze()
        test_loss = loss_fn(test_pred, test_y).item()

    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Test Loss: {test_loss:.4f}")

    # --- Early stopping check ---
    if test_loss < best_test_loss:
        best_test_loss   = test_loss
        best_weights     = {k: v.clone() for k, v in gnn_model_trained.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹ Early stopping at epoch {epoch+1} (best test loss: {best_test_loss:.4f})")
            break

# Restore best weights
gnn_model_trained.load_state_dict(best_weights)
print("✅ Training complete — best weights restored")


⏹ Early stopping at epoch 40 (best test loss: 0.1768)
✅ Training complete — best weights restored


In [15]:
# Continue training from where we left off — 100 more epochs
# gnn_model_trained.train()
# for epoch in range(100):
#     total_loss = 0
#     for x_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#         pred = gnn_model_trained(x_batch).squeeze()
#         loss = loss_fn(pred, y_batch)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()

#     scheduler.step()
#     if (epoch + 1) % 25 == 0:
#         print(f"  Epoch {epoch+1}/100 | Avg Loss: {total_loss/len(train_loader):.4f}")

# print("✅ Done")

In [16]:
# # ── Retrain with boundary-aware loss ─────────────────────────────────────────

# def weighted_loss(pred, target, threshold_norm, penalty=3.0):
#     """
#     Applies extra penalty when prediction crosses the busy/clear threshold
#     in the wrong direction.
#     """
#     base_loss = (pred - target) ** 2

#     # Identify samples near the threshold (within 1 std)
#     near_boundary = (torch.abs(target - threshold_norm) < 0.5).float()

#     # Extra penalty for crossing the boundary in the wrong direction
#     wrong_side = ((pred - threshold_norm) * (target - threshold_norm) < 0).float()

#     weights = 1.0 + (penalty - 1.0) * near_boundary * wrong_side
#     return (weights * base_loss).mean()

# # Threshold of 5 hrs in normalized space
# threshold_norm = (5.0 - y_mean) / y_std
# print(f"Threshold in normalized space: {threshold_norm:.3f}")

# # Fresh optimizer with lower LR for fine-tuning
# optimizer_ft = Adam(gnn_model_trained.parameters(), lr=0.0002, weight_decay=1e-4)

# gnn_model_trained.train()
# for epoch in range(200):
#     total_loss = 0
#     for x_batch, y_batch in train_loader:
#         optimizer_ft.zero_grad()
#         pred = gnn_model_trained(x_batch).squeeze()
#         loss = weighted_loss(pred, y_batch, threshold_norm)
#         loss.backward()
#         optimizer_ft.step()
#         total_loss += loss.item()

#     if (epoch + 1) % 50 == 0:
#         print(f"  Epoch {epoch+1}/200 | Avg Loss: {total_loss/len(train_loader):.4f}")

# print("✅ Fine-tuning complete")

In [17]:
# Check what threshold makes sense
import numpy as np

actuals = train_y_raw.numpy()
for t in [5, 8, 10, 12, 15]:
    pct_above = (actuals > t).mean() * 100
    print(f"Threshold {t:2d} hrs → {pct_above:.1f}% of ports flagged as BUSY")

Threshold  5 hrs → 59.4% of ports flagged as BUSY
Threshold  8 hrs → 36.4% of ports flagged as BUSY
Threshold 10 hrs → 36.4% of ports flagged as BUSY
Threshold 12 hrs → 36.4% of ports flagged as BUSY
Threshold 15 hrs → 34.4% of ports flagged as BUSY


In [18]:
# ── Cell 19 replacement ───────────────────────────────────────────────────────
threshold = 8.0

gnn_model_trained.eval()
with torch.no_grad():
    # ✅ Evaluate on TEST set — data the model has never seen
    pred_norm_test = gnn_model_trained(test_x).squeeze()
    pred_real_test = (pred_norm_test * y_std) + y_mean

    # Train predictions (for comparison — expect these to look better)
    pred_norm_train = gnn_model_trained(train_x).squeeze()
    pred_real_train = (pred_norm_train * y_std) + y_mean

def report(name, actual_raw, pred_real):
    actual = actual_raw.numpy()
    pred   = pred_real.numpy()
    correct = np.sum((pred > threshold) == (actual > threshold))
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"MAE             : {mean_absolute_error(actual, pred):.2f} hrs")
    print(f"RMSE            : {np.sqrt(mean_squared_error(actual, pred)):.2f} hrs")
    print(f"R² Score        : {r2_score(actual, pred):.4f}")
    print(f"Directional Acc : {correct/len(actual)*100:.1f}%  (threshold: {threshold} hrs)")

report("TRAIN SET (reference)", train_y_raw, pred_real_train)
report("TEST SET  (real score)", test_y_raw,  pred_real_test)

# ✅ Gap check — flag overfitting
train_r2 = r2_score(train_y_raw.numpy(), pred_real_train.numpy())
test_r2  = r2_score(test_y_raw.numpy(),  pred_real_test.numpy())
gap = train_r2 - test_r2
print(f"\nR² Gap (train - test): {gap:.4f}", end="  ")
if gap < 0.05:   print("🟢 No significant overfitting")
elif gap < 0.15: print("🟡 Mild overfitting — monitor")
else:            print("🔴 Overfitting — consider more Dropout or less depth")


  TRAIN SET (reference)
MAE             : 3.75 hrs
RMSE            : 5.65 hrs
R² Score        : 0.8294
Directional Acc : 100.0%  (threshold: 8.0 hrs)

  TEST SET  (real score)
MAE             : 3.80 hrs
RMSE            : 5.75 hrs
R² Score        : 0.8245
Directional Acc : 100.0%  (threshold: 8.0 hrs)

R² Gap (train - test): 0.0049  🟢 No significant overfitting


# **Prophet Model**

In [19]:
from prophet import Prophet
import pandas as pd
import matplotlib.pyplot as plt

# Prophet needs columns named exactly 'ds' and 'y'
# We'll train one model per port — Prophet works best on individual time series

prophet_source = port_hourly.copy()
prophet_source = prophet_source.reset_index()  # ds and port_id become columns

# Map port IDs to names for readability
prophet_source['port_name'] = prophet_source['port_id'].map(id_map)
prophet_source = prophet_source.dropna(subset=['port_name', 'waiting_time_hours'])

# Resample to daily average per port (Prophet works on daily granularity best)
prophet_daily = (
    prophet_source
    .groupby(['port_name', pd.Grouper(key='ds', freq='D')])['waiting_time_hours']
    .mean()
    .reset_index()
    .rename(columns={'waiting_time_hours': 'y'})
)

print(f"✅ Prophet data ready")
print(f"   Ports     : {prophet_daily['port_name'].unique().tolist()}")
print(f"   Date range: {prophet_daily['ds'].min()} → {prophet_daily['ds'].max()}")
print(f"   Total rows: {len(prophet_daily)}")

✅ Prophet data ready
   Ports     : ['Alexandria', 'Damietta', 'Dekheila', 'Port Said', 'Sokhna', 'Suez']
   Date range: 2024-01-01 00:00:00 → 2024-12-31 00:00:00
   Total rows: 2196


In [20]:
from prophet import Prophet

FORECAST_DAYS = 30   # how far ahead to forecast

prophet_models   = {}   # port_name → trained model
prophet_forecasts = {}  # port_name → forecast dataframe

for port in prophet_daily['port_name'].unique():
    port_df = (
        prophet_daily[prophet_daily['port_name'] == port][['ds', 'y']]
        .dropna()
        .sort_values('ds')
    )

    # Need enough history — skip ports with under 60 days
    if len(port_df) < 60:
        print(f"⚠️  {port}: not enough data ({len(port_df)} days), skipping")
        continue

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,    # already aggregated to daily
        seasonality_mode='multiplicative',  # better for shipping (spikes, not shifts)
        changepoint_prior_scale=0.05,       # conservative — avoids overfitting trends
        interval_width=0.80                 # 80% confidence interval
    )

    model.fit(port_df)

    future   = model.make_future_dataframe(periods=FORECAST_DAYS)
    forecast = model.predict(future)

    prophet_models[port]    = model
    prophet_forecasts[port] = forecast

    print(f"✅ {port}: trained on {len(port_df)} days → forecast {FORECAST_DAYS} days ahead")

print(f"\n✅ Done — {len(prophet_models)} port models trained")

06:05:59 - cmdstanpy - INFO - Chain [1] start processing
06:05:59 - cmdstanpy - INFO - Chain [1] done processing
06:06:00 - cmdstanpy - INFO - Chain [1] start processing
06:06:00 - cmdstanpy - INFO - Chain [1] done processing
06:06:01 - cmdstanpy - INFO - Chain [1] start processing
06:06:01 - cmdstanpy - INFO - Chain [1] done processing


✅ Alexandria: trained on 366 days → forecast 30 days ahead
✅ Damietta: trained on 366 days → forecast 30 days ahead


06:06:01 - cmdstanpy - INFO - Chain [1] start processing
06:06:01 - cmdstanpy - INFO - Chain [1] done processing
06:06:01 - cmdstanpy - INFO - Chain [1] start processing
06:06:01 - cmdstanpy - INFO - Chain [1] done processing


✅ Dekheila: trained on 366 days → forecast 30 days ahead
✅ Port Said: trained on 366 days → forecast 30 days ahead


06:06:01 - cmdstanpy - INFO - Chain [1] start processing
06:06:01 - cmdstanpy - INFO - Chain [1] done processing


✅ Sokhna: trained on 366 days → forecast 30 days ahead
✅ Suez: trained on 366 days → forecast 30 days ahead

✅ Done — 6 port models trained


In [21]:
# Add this before Cell B to see exactly how many days each port has
for port in prophet_daily['port_name'].unique():
    port_df = prophet_daily[prophet_daily['port_name'] == port].dropna()
    print(f"{port:12s}: {len(port_df)} days  ({port_df['ds'].min().date()} → {port_df['ds'].max().date()})")

Alexandria  : 366 days  (2024-01-01 → 2024-12-31)
Damietta    : 366 days  (2024-01-01 → 2024-12-31)
Dekheila    : 366 days  (2024-01-01 → 2024-12-31)
Port Said   : 366 days  (2024-01-01 → 2024-12-31)
Sokhna      : 366 days  (2024-01-01 → 2024-12-31)
Suez        : 366 days  (2024-01-01 → 2024-12-31)


In [22]:
print(port_hourly['port_id'].unique().tolist())

['EG_DMT', 'EG_PSD', 'EG_SAF', 'EG_SUZ', 'EG_ALY', 'EG_SOK']


In [24]:
for port, model in prophet_models.items():
    port_df = (
        prophet_daily[prophet_daily['port_name'] == port][['ds', 'y']]
        .dropna().sort_values('ds')
    )

    train_df_p = port_df.iloc[:-30]
    test_df_p  = port_df.iloc[-30:]

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05,
        interval_width=0.80
    )
    m.fit(train_df_p)

    future   = m.make_future_dataframe(periods=30)
    forecast = m.predict(future)

    preds  = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].set_index('ds')
    actuals = test_df_p.set_index('ds')
    merged  = actuals.join(preds, how='inner')

    if len(merged) == 0:
        continue

    mae = mean_absolute_error(merged['y'], merged['yhat'])

    # What % of actuals fall within the 80% confidence interval?
    within_ci = ((merged['y'] >= merged['yhat_lower']) & 
                 (merged['y'] <= merged['yhat_upper'])).mean() * 100

    # Is the trend direction correct? (rising/falling)
    actual_trend  = merged['y'].iloc[-1] > merged['y'].iloc[0]
    forecast_trend = merged['yhat'].iloc[-1] > merged['yhat'].iloc[0]
    trend_correct = actual_trend == forecast_trend

    print(f"\n{port}")
    print(f"  MAE              : {mae:.2f} hrs")
    print(f"  Within 80% CI    : {within_ci:.1f}%  (good if > 70%)")
    print(f"  Trend direction  : {'✅ Correct' if trend_correct else '❌ Wrong'}")

06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing
06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing
06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing



Alexandria
  MAE              : 3.77 hrs
  Within 80% CI    : 63.3%  (good if > 70%)
  Trend direction  : ❌ Wrong

Damietta
  MAE              : 2.46 hrs
  Within 80% CI    : 83.3%  (good if > 70%)
  Trend direction  : ❌ Wrong


06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing
06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing



Dekheila
  MAE              : 3.72 hrs
  Within 80% CI    : 60.0%  (good if > 70%)
  Trend direction  : ✅ Correct

Port Said
  MAE              : 3.18 hrs
  Within 80% CI    : 73.3%  (good if > 70%)
  Trend direction  : ✅ Correct


06:11:58 - cmdstanpy - INFO - Chain [1] start processing
06:11:58 - cmdstanpy - INFO - Chain [1] done processing



Sokhna
  MAE              : 3.18 hrs
  Within 80% CI    : 80.0%  (good if > 70%)
  Trend direction  : ✅ Correct

Suez
  MAE              : 3.63 hrs
  Within 80% CI    : 56.7%  (good if > 70%)
  Trend direction  : ✅ Correct


# **Saving the models**

In [27]:
import joblib
import json
from prophet.serialize import model_to_json

# ===== 1. Random Forest (already saved, but let's verify) =====
joblib.dump(rf_model, "rf_forecasting_model.joblib")
print("✅ Random Forest saved → rf_forecasting_model.joblib")

# ===== 2. GNN (PyTorch) =====
torch.save(gnn_model_trained.state_dict(), "gnn_model_weights.pth")
print("✅ GNN weights saved → gnn_model_weights.pth")

# ===== 3. Prophet (all 6 port models) =====
prophet_models_serialized = {}
for port_name, model in prophet_models.items():
    with open(f"prophet_model_{port_name}.json", "w") as f:
        f.write(model_to_json(model))
    prophet_models_serialized[port_name] = f"prophet_model_{port_name}.json"
    print(f"   ✅ Saved prophet_model_{port_name}.json")

✅ Random Forest saved → rf_forecasting_model.joblib
✅ GNN weights saved → gnn_model_weights.pth
   ✅ Saved prophet_model_Alexandria.json
   ✅ Saved prophet_model_Damietta.json
   ✅ Saved prophet_model_Dekheila.json
   ✅ Saved prophet_model_Port Said.json
   ✅ Saved prophet_model_Sokhna.json
   ✅ Saved prophet_model_Suez.json


In [29]:
# Save scalers and constants as JSON for easy loading in Node.js/Python backend
preprocessing_config = {
    "rf_scaler": {
        "mean": scaler_gnn.mean_.tolist(),  # From Cell 11
        "std": scaler_gnn.scale_.tolist()
    },
    "gnn_normalization": {
        "y_mean": float(y_mean),
        "y_std": float(y_std),
        "input_features": gnn_feature_cols
    },
    "rf_features": features,
    "port_mapping": id_map,
    "available_ports": available_ports,
    "threshold_busy_hours": 8.0
}

with open("preprocessing_config.json", "w") as f:
    json.dump(preprocessing_config, f, indent=2)

print("✅ Preprocessing config saved → preprocessing_config.json")

✅ Preprocessing config saved → preprocessing_config.json
